# Module 7: Synthetic Data Generation and RAG Evaluation with Ragas

In this notebook, we'll go end-to-end from **generating synthetic evaluation data** to **systematically evaluating and improving a RAG pipeline** — all using [Ragas](https://github.com/explodinggradients/ragas).

The flow is:
1. **Generate** synthetic test data using Ragas' knowledge graph-based approach
2. **Build** a baseline RAG application with LangChain and LangGraph
3. **Evaluate** the RAG application against our synthetic test set using Ragas metrics
4. **Iterate** on the pipeline and measure the impact

> **NOTE:** Ragas is framework-agnostic — while this example uses LangChain/LangGraph, you can use Ragas with any framework (or none at all). Ragas is best suited for finding *directional* changes in your LLM-based systems. The absolute scores aren't comparable in a vacuum.

## Outline

**Part 1: Synthetic Data Generation**
- Task 1: Dependencies and API Keys
- Task 2: Data Preparation
- Task 3: Knowledge Graph Construction
- Task 4: Generating Synthetic Test Data
- ***❓ Question #1 & Question #2***
- ***🏗️ Activity #1: Custom Query Distribution***

**Part 2: RAG Evaluation with Ragas**
- Task 5: Building a Baseline RAG Application
- Task 6: Evaluating with Ragas
- Task 7: Making Adjustments and Re-Evaluating
- ***❓ Question #3, Question #4, Question #5, & Question #6***
- ***🏗️ Activity #2: Implement a Different Reranking Strategy***

---
# Part 1: Synthetic Data Generation with Ragas

Before we can evaluate a RAG system, we need high-quality test data. Manually creating question-answer pairs is time-consuming and often biased toward simple queries. Ragas solves this by building a **knowledge graph** from your documents and using it to generate diverse, realistic test questions automatically.

We'll use the **Stone Ridge 2025 Investor Letter** and an **Alternative Investments Handbook** as our source documents — maintaining continuity with the investment advisory use case from previous sessions.

## Task 1: Dependencies and API Keys

If you have not already done so, install the required libraries using the uv package manager:
```bash
uv sync
```

We'll need API keys for:
- **OpenAI** — for LLM and embedding models (used in both SDG and RAG evaluation)
- **Cohere** — for reranking in the improved pipeline ([sign up here](https://docs.cohere.com/reference/about))

You have two options for supplying your API keys:
- Use environment variables (copy `.env.sample` to `.env` and fill in your keys)
- Provide them via the prompts below

In [1]:
import nest_asyncio
nest_asyncio.apply()

In [2]:
import nltk
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/yitaek.hwang/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/yitaek.hwang/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.


True

In [49]:
import os
from getpass import getpass
from dotenv import load_dotenv

load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Please enter your OpenAI API key!")

if not os.environ.get("COHERE_API_KEY"):
    os.environ["COHERE_API_KEY"] = getpass("Please enter your Cohere API key!")

## Task 2: Data Preparation

We'll prepare our data using two complementary investment-focused sources:
- **Stone Ridge 2025 Investor Letter** — covering Stone Ridge's investment philosophy, Bayesian approach to decision-making, energy investments, reinsurance, and risk management
- **Alternative Investments Handbook** — covering alternative asset classes including real estate, private equity, hedge funds, reinsurance, commodities, and infrastructure

The topical overlap between these documents (particularly around reinsurance, risk premiums, diversification, and alternative investments) helps Ragas build rich cross-document relationships in the knowledge graph.

In [50]:
from langchain_community.document_loaders import PyMuPDFLoader, TextLoader

# Load the Stone Ridge 2025 Investor Letter (PDF)
pdf_loader = PyMuPDFLoader("data/Stone Ridge 2025 Investor Letter.pdf")
pdf_docs = pdf_loader.load()

# Load the Alternative Investments Handbook (text)
txt_loader = TextLoader("data/AlternativeInvestmentsHandbook.txt")
txt_docs = txt_loader.load()

# Combine into a single list
docs = pdf_docs + txt_docs
print(f"Loaded {len(docs)} documents:")
print(f"  - Stone Ridge 2025 Investor Letter: {len(pdf_docs)} pages")
print(f"  - AlternativeInvestmentsHandbook.txt: {len(txt_docs)} document(s)")

Loaded 15 documents:
  - Stone Ridge 2025 Investor Letter: 14 pages
  - AlternativeInvestmentsHandbook.txt: 1 document(s)


## Task 3: Knowledge Graph Construction

Ragas uses a **knowledge graph-based approach** to create synthetic test data. This is powerful because it allows us to create complex, multi-hop queries — not just simple factoid questions. Systems tend to perform well on simple evaluation tasks, so this additional complexity helps us find real weaknesses.

The process works in three stages:
1. **Build the graph** — insert documents as nodes
2. **Apply transformations** — extract headlines, summaries, themes, entities, and embeddings
3. **Create relationships** — use cosine similarity and overlap scores to connect related nodes

Let's start by defining our `generator_llm` and `generator_embeddings`.

In [51]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-nano"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

### Step 1: Initialize the Knowledge Graph

We create an empty knowledge graph and populate it with our document nodes. Each full document becomes a node of type `DOCUMENT`.

In [52]:
from ragas.testset.graph import KnowledgeGraph, Node, NodeType

kg = KnowledgeGraph()

for doc in docs:
    kg.nodes.append(
        Node(
            type=NodeType.DOCUMENT,
            properties={"page_content": doc.page_content, "document_metadata": doc.metadata}
        )
    )
kg

KnowledgeGraph(nodes: 15, relationships: 0)

### Step 2: Apply Transformations

Now we apply the [default transformations](https://docs.ragas.io/en/latest/references/transforms/#ragas.testset.transforms.default_transforms) to enrich our knowledge graph. These transformations:

- **HeadlinesExtractor** — finds the overall headlines for each document
- **SummaryExtractor** — produces summaries of the documents
- **ThemesExtractor** — extracts broad themes
- **EmbeddingExtractor** — creates embeddings for similarity computation
- **NERExtractor** — extracts named entities

These are then used to build relationships between nodes via cosine similarity and overlap scoring.

In [53]:
from ragas.testset.transforms import default_transforms, apply_transforms

transforms = default_transforms(
    documents=docs,
    llm=generator_llm,
    embedding_model=generator_embeddings
)
apply_transforms(kg, transforms)
kg

Applying HeadlinesExtractor:   0%|          | 0/14 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/15 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node


Applying SummaryExtractor:   0%|          | 0/25 [00:00<?, ?it/s]

Property 'summary' already exists in node '2db385'. Skipping!
Property 'summary' already exists in node '5bb9b5'. Skipping!
Property 'summary' already exists in node 'b61634'. Skipping!
Property 'summary' already exists in node 'e46f0d'. Skipping!
Property 'summary' already exists in node '078d76'. Skipping!
Property 'summary' already exists in node 'd33ea7'. Skipping!
Property 'summary' already exists in node 'ca0164'. Skipping!
Property 'summary' already exists in node '6e2f7d'. Skipping!
Property 'summary' already exists in node '0c8363'. Skipping!
Property 'summary' already exists in node 'cb09ce'. Skipping!
Property 'summary' already exists in node 'e4dabf'. Skipping!


Applying CustomNodeFilter:   0%|          | 0/10 [00:00<?, ?it/s]

Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/43 [00:00<?, ?it/s]

Property 'summary_embedding' already exists in node '5bb9b5'. Skipping!
Property 'summary_embedding' already exists in node '2db385'. Skipping!
Property 'summary_embedding' already exists in node '6e2f7d'. Skipping!
Property 'summary_embedding' already exists in node 'e46f0d'. Skipping!
Property 'summary_embedding' already exists in node 'b61634'. Skipping!
Property 'summary_embedding' already exists in node 'ca0164'. Skipping!
Property 'summary_embedding' already exists in node '0c8363'. Skipping!
Property 'summary_embedding' already exists in node '078d76'. Skipping!
Property 'summary_embedding' already exists in node 'd33ea7'. Skipping!
Property 'summary_embedding' already exists in node 'cb09ce'. Skipping!
Property 'summary_embedding' already exists in node 'e4dabf'. Skipping!


Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

KnowledgeGraph(nodes: 35, relationships: 303)

### Step 3: Save the Knowledge Graph

Knowledge graphs can be saved and loaded, which is useful for iterating on test generation without rebuilding the graph each time.

In [54]:
kg.save("investment_data_kg.json")

# You can reload it later:
# kg = KnowledgeGraph.load("investment_data_kg.json")

## Task 4: Generating Synthetic Test Data

With our knowledge graph built, we can now generate synthetic test data. Ragas provides several **query synthesizers**, each producing a different type of question:

- **`SingleHopSpecificQuerySynthesizer`** — creates questions answerable from a single chunk of context (e.g., *"What is Stone Ridge's approach to reinsurance investing?"*)
- **`MultiHopAbstractQuerySynthesizer`** — creates questions requiring synthesis across multiple chunks at an abstract level (e.g., *"How do alternative risk premiums relate to portfolio diversification?"*)
- **`MultiHopSpecificQuerySynthesizer`** — creates questions requiring specific details from multiple chunks (e.g., *"How does Stone Ridge's Bayesian philosophy connect to their energy investment strategy?"*)

We define a **query distribution** to control the mix of question types.

In [55]:
from ragas.testset.synthesizers import (
    SingleHopSpecificQuerySynthesizer,
    MultiHopAbstractQuerySynthesizer,
    MultiHopSpecificQuerySynthesizer,
)

query_distribution = [
    (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 0.5),
    (MultiHopAbstractQuerySynthesizer(llm=generator_llm), 0.25),
    (MultiHopSpecificQuerySynthesizer(llm=generator_llm), 0.25),
]

In [56]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(
    llm=generator_llm,
    embedding_model=generator_embeddings,
    knowledge_graph=kg
)

testset = generator.generate(testset_size=10, query_distribution=query_distribution)
testset.to_pandas()

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/11 [00:00<?, ?it/s]

,user_input,reference_contexts,reference,synthesizer_name
0,How has the city of Chicago influenced your ca...,[and my career has essentially been a three-de...,My career has essentially been a three-decade ...,single_hop_specifc_query_synthesizer
1,How did Fisher exemplify Bayesian principles i...,"[TODAY’S THE DAY Fancy math aside, the foundat...",Fisher exemplified Bayesian principles by usin...,single_hop_specifc_query_synthesizer
2,What is the recent performance of Stone Ridge ...,[Standardized returns as of most recent quarte...,Stone Ridge Energy Equity Tranches had a stand...,single_hop_specifc_query_synthesizer
3,What information does Chapter 1 of The Alterna...,[The Alternative Investments Handbook A Practi...,Chapter 1 explains that alternative investment...,single_hop_specifc_query_synthesizer
4,What NOI means in real estate?,[PART 2: REAL ESTATE INVESTMENTS Chapter 4: Re...,"NOI stands for Net Operating Income, which is ...",single_hop_specifc_query_synthesizer
5,How do commodities and real assets serve as an...,[<1-hop>\n\nPART 6: COMMODITIES AND REAL ASSET...,Commodities and real assets serve as an inflat...,multi_hop_abstract_query_synthesizer
6,How does the central bank demand for gold rela...,[<1-hop>\n\nPART 6: COMMODITIES AND REAL ASSET...,"The central bank demand for gold, as part of t...",multi_hop_abstract_query_synthesizer
7,How learning from experience and updating beli...,[<1-hop>\n\nand my career has essentially been...,The context explains that Bayesian treasure hu...,multi_hop_abstract_query_synthesizer
8,How do Chapter 1's overview of alternative ass...,[<1-hop>\n\nPART 2: REAL ESTATE INVESTMENTS Ch...,Chapter 1 introduces alternative investments s...,multi_hop_specific_query_synthesizer
9,"In context of Chapter 4 and Chapter 13, how do...",[<1-hop>\n\nPART 6: COMMODITIES AND REAL ASSET...,The context from Chapter 4 discusses real esta...,multi_hop_specific_query_synthesizer


### Abstracted SDG (Shortcut)

The above was the **unrolled** process showing each step. Ragas also provides a one-liner that builds the knowledge graph under the hood and generates the test set in a single call. This is convenient for quick iteration:

In [57]:
# Abstracted approach (for reference):
# generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
# dataset = generator.generate_with_langchain_docs(docs, testset_size=10)

### ❓ Question #1:

What are the three types of query synthesizers doing? Describe each one in simple terms.

##### ✅ Answer:

The three types of query synthesizers generate different kinds of test questions:

1. **SingleHopSpecificQuerySynthesizer**: Creates straightforward questions that can be answered by looking at a single chunk of text. These are "simple lookup" questions that don't require connecting information across multiple parts of the document. For example: "What is Stone Ridge's approach to reinsurance investing?"

2. **MultiHopAbstractQuerySynthesizer**: Creates complex questions that require combining information from multiple chunks at a conceptual or abstract level. These questions test whether the RAG system can synthesize themes and high-level concepts across different parts of the documents. For example: "How do alternative risk premiums relate to portfolio diversification?"

3. **MultiHopSpecificQuerySynthesizer**: Creates questions that require specific details from multiple chunks of text. Unlike the abstract version, these questions need precise facts from different sections to be combined. For example: "How does Stone Ridge's Bayesian philosophy connect to their energy investment strategy?"

The mix of these three types helps create a comprehensive test suite that challenges the RAG system in different ways.

### ❓ Question #2:

Ragas offers both an "unrolled" (manual) approach and an "abstracted" (automatic) approach to synthetic data generation. What are the trade-offs between these two approaches? When would you choose one over the other?

##### ✅ Answer:

**Unrolled (Manual) Approach:**

*Advantages:*
- **Full control and transparency**: You can see and customize each step of the knowledge graph construction
- **Inspectable**: You can examine the knowledge graph at any stage to understand the relationships being built
- **Reusable**: You can save the knowledge graph and reload it later, avoiding expensive rebuilding
- **Customizable**: You can modify or add custom transformations, adjust parameters, or even implement custom relationship builders
- **Debuggable**: Easier to identify where issues occur in the pipeline

*Disadvantages:*
- **More verbose**: Requires more code and understanding of the internal process
- **Steeper learning curve**: Need to understand each transformation and how they work together
- **Time investment**: Takes longer to set up initially

**Abstracted (Automatic) Approach:**

*Advantages:*
- **Quick and easy**: Single line of code to generate test data
- **Good for prototyping**: Fast iteration when experimenting with different datasets
- **Lower barrier to entry**: Don't need deep understanding of internal mechanics

*Disadvantages:*
- **Black box**: Can't inspect or modify the knowledge graph creation process
- **Less control**: Stuck with default transformations and parameters
- **Not reusable**: Can't save and reload the knowledge graph, must regenerate each time
- **Harder to debug**: If something goes wrong, it's difficult to pinpoint the issue

**When to choose each:**

- **Choose Unrolled** when: You need to iterate on the same dataset multiple times, want to customize the knowledge graph construction, need to debug issues, or want to understand the underlying process deeply
- **Choose Abstracted** when: You're doing quick experiments, testing different datasets, don't need customization, or are just getting started with Ragas

### 🏗️ Activity #1: Custom Query Distribution

Modify the `query_distribution` to experiment with different ratios of query types.

**Requirements:**
1. Create a custom query distribution with different weights than the default
2. Generate a new test set using your custom distribution
3. Compare the types of questions generated with the default distribution
4. Explain why you chose the weights you did

In [ ]:

custom_query_distribution = [
    (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 0.3),  # Reduced from 0.5
    (MultiHopAbstractQuerySynthesizer(llm=generator_llm), 0.4),   # Increased from 0.25
    (MultiHopSpecificQuerySynthesizer(llm=generator_llm), 0.3),   # Increased from 0.25
]

# Generate a new test set using the custom distribution
custom_generator = TestsetGenerator(
    llm=generator_llm,
    embedding_model=generator_embeddings,
    knowledge_graph=kg
)

custom_testset = custom_generator.generate(
    testset_size=10,
    query_distribution=custom_query_distribution
)

# Display the results
custom_df = custom_testset.to_pandas()
print("Custom Query Distribution Results:")
print("=" * 80)
print(f"Total questions generated: {len(custom_df)}")
print("\nBreakdown by synthesizer type:")
print(custom_df['synthesizer_name'].value_counts())
print("\nSample questions:")
for idx, row in custom_df.head(5).iterrows():
    print(f"\n{idx + 1}. {row['user_input']}")
    print(f"   Type: {row['synthesizer_name']}")

# Compare with default distribution
print("\n" + "=" * 80)
print("COMPARISON:")
print("=" * 80)
print("\nDefault Distribution (50/25/25):")
print("- Emphasizes simple lookup questions")
print("- Good for testing basic retrieval")
print("\nCustom Distribution (30/40/30):")
print("- Emphasizes complex synthesis across documents")
print("- Better reflects real-world investment advisory queries")
print("- More challenging for RAG systems to answer correctly")
print("\nRationale for custom weights:")
print("Investment advisory use cases typically require synthesizing information")
print("across multiple sources (e.g., combining market trends, risk analysis,")
print("and historical performance). The custom distribution better reflects this")
print("by generating more multi-hop queries that test the RAG system's ability")
print("to connect concepts and provide comprehensive answers.")

---
# Part 2: RAG Evaluation with Ragas

Now that we have our synthetic test data, we can use it to **systematically evaluate** a RAG pipeline. The idea is simple:
1. Build a RAG application
2. Run our synthetic queries through it
3. Score the results using Ragas metrics
4. Make changes and measure the impact

This gives us a **data-driven approach** to improving our RAG system, rather than relying on vibes.

## Task 5: Building a Baseline RAG Application

We'll build a deliberately simple (and somewhat bad) RAG pipeline as our **baseline**, so we can clearly see the impact of improvements later.

Our baseline uses:
- Tiny chunks (50 characters) with no overlap
- A small embedding model (`text-embedding-3-small`)
- Only 3 retrieved documents
- A basic prompt

> **NOTE:** We use the same data that our synthetic test set was generated from — this is required because the test questions are specifically designed for this investment data.

### R — Retrieval

First, we chunk our documents and build a vector store.

In [59]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=50, chunk_overlap=0)
split_documents = text_splitter.split_documents(docs)
len(split_documents)

2045

### ❓ Question #3:

What is the purpose of the `chunk_overlap` parameter in the `RecursiveCharacterTextSplitter`?

##### ✅ Answer:

The `chunk_overlap` parameter controls how many characters from the end of one chunk are duplicated at the beginning of the next chunk. This serves several important purposes:

1. **Preserves Context Across Boundaries**: When documents are split into chunks, important information or concepts might naturally span across what would be a chunk boundary. The overlap ensures that this information is captured completely in at least one chunk.

2. **Prevents Information Loss**: Without overlap, a sentence or paragraph that happens to be split between two chunks might lose its meaning. If someone is searching for information that spans that boundary, they might not retrieve the relevant context. The overlap acts as a buffer to prevent this.

3. **Improves Retrieval Quality**: When a query relates to information near a chunk boundary, the overlap increases the chances that the semantic similarity search will retrieve at least one chunk with the complete, relevant context.

4. **Trade-offs**: 
   - **No overlap (0)**: Maximum efficiency, no duplicate text, but risk of splitting important context
   - **Small overlap (20-50 chars)**: Good balance for most use cases
   - **Large overlap (100+ chars)**: Better context preservation but more storage and processing overhead

In the baseline example, `chunk_overlap=0` was used deliberately to create a weaker system. In the improved version, `chunk_overlap=30` was added to better preserve context across chunk boundaries.

In [60]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

In [61]:
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams

client = QdrantClient(":memory:")

client.create_collection(
    collection_name="baseline_rag",
    vectors_config=VectorParams(size=1536, distance=Distance.COSINE),
)

vector_store = QdrantVectorStore(
    client=client,
    collection_name="baseline_rag",
    embedding=embeddings,
)

_ = vector_store.add_documents(documents=split_documents)

In [62]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

In [63]:
def retrieve(state):
    retrieved_docs = retriever.invoke(state["question"])
    return {"context": retrieved_docs}

### A — Augmented

A simple RAG prompt:

In [64]:
from langchain_core.prompts import ChatPromptTemplate

RAG_PROMPT = """\
You are a helpful investment advisory assistant who answers questions based on provided context. You must only use the provided context, and cannot use your own knowledge.

### Question
{question}

### Context
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_PROMPT)

### G — Generation

We use `gpt-4.1-nano` for generation to avoid using the same model as our judge model.

In [65]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4.1-nano")

In [66]:
def generate(state):
    docs_content = "\n\n".join(doc.page_content for doc in state["context"])
    messages = rag_prompt.format_messages(question=state["question"], context=docs_content)
    response = llm.invoke(messages)
    return {"response": response.content}

### Building the RAG Graph with LangGraph

In [67]:
from langgraph.graph import START, StateGraph
from typing_extensions import List, TypedDict
from langchain_core.documents import Document

class State(TypedDict):
    question: str
    context: List[Document]
    response: str

In [68]:
graph_builder = StateGraph(State).add_sequence([retrieve, generate])
graph_builder.add_edge(START, "retrieve")
graph = graph_builder.compile()

Let's do a quick sanity check:

In [69]:
response = graph.invoke({"question": "What is Stone Ridge's investment philosophy?"})
response["response"]

"Stone Ridge's investment philosophy is centered on a relentless focus on growing."

With tiny 50-character chunks and only 3 retrieved documents, the baseline likely struggles to provide good answers about Stone Ridge's investment philosophy. That's intentional — it gives us room to improve!

## Task 6: Evaluating with Ragas

Now we can evaluate our baseline RAG against the synthetic test data we generated in Part 1.

First, we run all the synthetic queries through our RAG pipeline to collect responses and retrieved contexts.

In [70]:
for test_row in testset:
    response = graph.invoke({"question": test_row.eval_sample.user_input})
    test_row.eval_sample.response = response["response"]
    test_row.eval_sample.retrieved_contexts = [context.page_content for context in response["context"]]

Convert to an `EvaluationDataset` for smoother evaluation:

In [71]:
from ragas import EvaluationDataset

evaluation_dataset = EvaluationDataset.from_pandas(testset.to_pandas())

We select a **judge model** — a separate, capable model that scores the outputs. Using a different model than the generator avoids self-evaluation bias.

In [72]:
from ragas.llms import LangchainLLMWrapper

evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini"))

### Running the Baseline Evaluation

We evaluate across six metrics:
- **Context Recall** — did we retrieve the relevant context?
- **Faithfulness** — is the answer grounded in the retrieved context?
- **Factual Correctness** — is the answer factually correct vs. the reference?
- **Answer Relevancy** — is the answer relevant to the question?
- **Context Entity Recall** — did we capture the key entities from the reference context?
- **Noise Sensitivity** — is the answer affected by irrelevant retrieved content?

In [73]:
from ragas.metrics import (
    LLMContextRecall,
    Faithfulness,
    FactualCorrectness,
    ResponseRelevancy,
    ContextEntityRecall,
    NoiseSensitivity,
)
from ragas import evaluate, RunConfig

custom_run_config = RunConfig(timeout=360)

baseline_result = evaluate(
    dataset=evaluation_dataset,
    metrics=[
        LLMContextRecall(),
        Faithfulness(),
        FactualCorrectness(),
        ResponseRelevancy(),
        ContextEntityRecall(),
        NoiseSensitivity(),
    ],
    llm=evaluator_llm,
    run_config=custom_run_config,
)
baseline_result

Evaluating:   0%|          | 0/66 [00:00<?, ?it/s]

Exception raised in Job[30]: AttributeError('StringIO' object has no attribute 'classifications')
Exception raised in Job[37]: AttributeError('StringIO' object has no attribute 'statements')


{'context_recall': 0.1433, 'faithfulness': 0.3754, 'factual_correctness': 0.5409, 'answer_relevancy': 0.6851, 'context_entity_recall': 0.1887, 'noise_sensitivity_relevant': 0.0000}

## Task 7: Making Adjustments and Re-Evaluating

Now that we have a baseline, let's improve the pipeline and measure the impact. We'll make three changes:

1. **Larger chunks** (500 characters with 30 overlap instead of 50 with 0 overlap)
2. **More documents retrieved** (k=20 instead of k=3)
3. **Reranking with Cohere** — retrieves 20 documents, then uses Cohere's reranker to select the top 5

Reranking is a technique that uses a cross-encoder model (slower but more accurate than embedding similarity) on a smaller subset of candidates to improve retrieval precision.

In [74]:
from langchain_openai import OpenAIEmbeddings
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=30)
split_documents = text_splitter.split_documents(docs)
print(f"Improved chunking: {len(split_documents)} chunks (vs baseline)")

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

client = QdrantClient(":memory:")
client.create_collection(
    collection_name="improved_rag",
    vectors_config=VectorParams(size=1536, distance=Distance.COSINE),
)

vector_store = QdrantVectorStore(
    client=client,
    collection_name="improved_rag",
    embedding=embeddings,
)

_ = vector_store.add_documents(documents=split_documents)
adjusted_retriever = vector_store.as_retriever(search_kwargs={"k": 20})

Improved chunking: 202 chunks (vs baseline)


In [75]:
from langchain_classic.retrievers.contextual_compression import (
    ContextualCompressionRetriever,
)
from langchain_cohere import CohereRerank

def retrieve_adjusted(state):
    compressor = CohereRerank(model="rerank-v3.5")
    compression_retriever = ContextualCompressionRetriever(
        base_compressor=compressor,
        base_retriever=adjusted_retriever,
        search_kwargs={"k": 5},
    )
    retrieved_docs = compression_retriever.invoke(state["question"])
    return {"context": retrieved_docs}

In [76]:
from typing import TypedDict, List
from langchain_core.documents import Document

class AdjustedState(TypedDict):
    question: str
    context: List[Document]
    response: str

adjusted_graph_builder = StateGraph(AdjustedState).add_sequence([retrieve_adjusted, generate])
adjusted_graph_builder.add_edge(START, "retrieve_adjusted")
adjusted_graph = adjusted_graph_builder.compile()

Let's verify the improved pipeline works:

In [77]:
response = adjusted_graph.invoke({"question": "How does Stone Ridge approach risk management in their energy investments?"})
response["response"]

'The provided context does not include specific details on how Stone Ridge approaches risk management in their energy investments.'

### Running the Improved Evaluation

Now let's run the same synthetic test set through our improved pipeline and compare.

In [78]:
import time
import copy

rerank_testset = copy.deepcopy(testset)

for test_row in rerank_testset:
    response = adjusted_graph.invoke({"question": test_row.eval_sample.user_input})
    test_row.eval_sample.response = response["response"]
    test_row.eval_sample.retrieved_contexts = [context.page_content for context in response["context"]]
    time.sleep(2)  # To avoid rate limiting

In [46]:
rerank_evaluation_dataset = EvaluationDataset.from_pandas(rerank_testset.to_pandas())

rerank_result = evaluate(
    dataset=rerank_evaluation_dataset,
    metrics=[
        LLMContextRecall(),
        Faithfulness(),
        FactualCorrectness(),
        ResponseRelevancy(),
        ContextEntityRecall(),
        NoiseSensitivity(),
    ],
    llm=evaluator_llm,
    run_config=custom_run_config,
)
rerank_result

Evaluating:   0%|          | 0/66 [00:00<?, ?it/s]

{'context_recall': 0.6833, 'faithfulness': 0.5825, 'factual_correctness': 0.5364, 'answer_relevancy': 0.7742, 'context_entity_recall': 0.3159, 'noise_sensitivity_relevant': 0.1507}

### ❓ Question #4:

Which system performed better, on what metrics, and why?

##### ✅ Answer:

The **improved system with reranking** performed significantly better on most metrics:

**Performance Comparison:**

| Metric | Baseline | Improved (Reranking) | Change |
|--------|----------|---------------------|---------|
| Context Recall | 0.1433 | 0.6833 | +376% ⬆️ |
| Faithfulness | 0.3754 | 0.5825 | +55% ⬆️ |
| Factual Correctness | 0.5409 | 0.5364 | -1% (minimal) |
| Answer Relevancy | 0.6851 | 0.7742 | +13% ⬆️ |
| Context Entity Recall | 0.1887 | 0.3159 | +67% ⬆️ |
| Noise Sensitivity | 0.0000 | 0.1507 | +15% ⬆️ |

**Key Improvements:**

1. **Context Recall (most dramatic improvement)**: The improved system retrieves far more relevant context. This makes sense because:
   - Larger chunks (500 vs 50 chars) capture more complete information
   - More documents retrieved (k=20 vs k=3) gives more candidates
   - Reranking selects the most relevant 5 from the 20 candidates

2. **Faithfulness**: Better context leads to more grounded responses that stick to the retrieved information rather than hallucinating.

3. **Answer Relevancy**: Improved slightly because better context enables more focused, relevant answers.

4. **Context Entity Recall**: Larger chunks naturally capture more entities mentioned in reference contexts.

5. **Noise Sensitivity**: Interestingly, this increased (which is good - it means the system is better at handling irrelevant information).

**Why the Improvements Worked:**

- **Chunk size**: 50-character chunks were absurdly small and would split sentences mid-word. 500-character chunks capture complete thoughts and context.
- **Chunk overlap**: Adding 30 characters of overlap prevents information loss at boundaries.
- **Higher k value**: Retrieving 20 candidates instead of 3 gives the reranker more options to work with.
- **Cohere reranking**: Cross-encoder models are more accurate than embedding similarity because they can directly compare the query to each document, rather than relying on vector similarity alone.

**Factual Correctness remained similar** because both systems struggled to provide factually correct answers - the baseline because it lacked context, the improved system possibly because the LLM still needs better context or a better generation prompt.

### ❓ Question #5:

What are the benefits and limitations of using synthetic data generation for RAG evaluation? Consider both the practical advantages and potential pitfalls.

##### ✅ Answer:

**Benefits:**

1. **Speed and Scale**: 
   - Generate hundreds of test cases in minutes vs. days/weeks of manual creation
   - Enables rapid iteration during development
   - Cost-effective compared to hiring subject matter experts to write questions

2. **Diversity and Coverage**:
   - Knowledge graph-based approach creates questions you might not think to ask manually
   - Ensures coverage across different document sections and relationship types
   - Can generate different complexity levels (single-hop, multi-hop) systematically

3. **Reproducibility**:
   - Same source documents + same parameters = same test set
   - Easier to track performance changes over time
   - Facilitates A/B testing and controlled experiments

4. **Reduces Human Bias**:
   - Manual test creation tends toward "happy path" scenarios
   - Humans often create overly simple questions
   - Synthetic generation explores edge cases and complex relationships

5. **Bootstrapping**:
   - Provides immediate evaluation capability for new domains
   - Don't need to wait for real user queries to accumulate
   - Helps identify major issues early in development

**Limitations:**

1. **Distribution Mismatch**:
   - Synthetic queries may not reflect real user questions and language patterns
   - Real users ask questions in unexpected ways, use slang, make typos, or are vague
   - May over-optimize for synthetic patterns at the expense of real-world performance

2. **Quality Ceiling**:
   - Limited by the quality of the generator LLM
   - Can't generate questions requiring knowledge beyond the source documents
   - May miss domain-specific nuances or edge cases

3. **Ground Truth Challenges**:
   - Reference answers are also LLM-generated, not human-verified
   - Risk of "teaching to the test" when both generation and evaluation use LLMs
   - May not catch subtle incorrectness if the generator makes the same mistakes

4. **False Confidence**:
   - Good scores on synthetic data don't guarantee production success
   - Real users have different expectations and tolerance for errors
   - Missing the "unknown unknowns" that only emerge with real usage

5. **Domain Limitations**:
   - Works best for factual, knowledge-based domains
   - Struggles with subjective queries (opinions, recommendations, creative tasks)
   - May not capture queries requiring real-time information or external context

6. **Computational Cost**:
   - Building knowledge graphs and generating test sets requires significant API calls
   - Can be expensive at scale
   - Need to balance test set size with cost

**Best Practices to Mitigate Limitations:**

- Use synthetic data for **directional improvement** (is change A better than B?) rather than absolute quality assessment
- Supplement with **human-created test cases** for critical paths
- **Validate with real user queries** as soon as possible
- Regularly **audit synthetic test sets** to ensure they remain relevant
- Use **multiple evaluation approaches** (synthetic + real + human evaluation)

### ❓ Question #6:

If you were building a production investment advisory assistant for Stone Ridge, which Ragas metrics would be most important to optimize for and why? Consider the financial services domain specifically.

##### ✅ Answer:

For a production investment advisory assistant in financial services, I would prioritize these metrics in order:

**1. Faithfulness (CRITICAL - Highest Priority)**
- **Why**: In financial services, making claims not supported by source documents can lead to:
  - Regulatory violations (SEC, FINRA requirements for substantiation)
  - Legal liability if clients make investment decisions based on hallucinated information
  - Loss of trust and reputation damage
  - Potential financial harm to clients
- **Target**: Should aim for >0.95 (95%+) faithfulness
- **Mitigation**: Implement citation requirements, show source documents, add human review for high-stakes queries

**2. Factual Correctness (CRITICAL - Highest Priority)**
- **Why**: Incorrect investment information can lead to:
  - Poor investment decisions and financial losses
  - Compliance violations
  - Loss of client trust
  - Legal exposure
- **Target**: Should aim for >0.90 (90%+)
- **Mitigation**: Add fact-checking layers, require citations, implement guardrails for financial figures and dates

**3. Context Recall (HIGH Priority)**
- **Why**: Investment decisions require complete information:
  - Missing relevant risk factors could lead to unsuitable recommendations
  - Incomplete fee disclosures violate transparency requirements
  - Omitting performance caveats can mislead clients
- **Target**: Should aim for >0.80 (80%+)
- **Financial Services Context**: Better to over-retrieve and include disclaimers than miss critical information
- **Example**: If discussing returns, must also retrieve and present risk factors and past performance disclaimers

**4. Context Entity Recall (MEDIUM-HIGH Priority)**
- **Why**: Financial documents contain critical entities:
  - Specific investment products and tickers
  - Performance figures and dates
  - Regulatory requirements and definitions
- **Target**: Should aim for >0.75 (75%+)
- **Example**: If discussing "SREF" (Stone Ridge Reinsurance Risk Premium Interval Fund), all mentions of this specific entity should be captured

**5. Answer Relevancy (MEDIUM Priority)**
- **Why**: Client time is valuable, and vague answers reduce utility
- **However**: In financial services, being thorough with disclosures is more important than being concise
- **Balance**: Answers should be relevant but err on the side of completeness
- **Target**: Should aim for >0.75 (75%+)

**6. Noise Sensitivity (LOWER Priority)**
- **Why**: While important, this is less critical than the above
- **Financial Services Context**: It's better to include potentially irrelevant disclaimers than to omit them
- **However**: Should still monitor to ensure users aren't confused by irrelevant information

**Additional Metrics NOT in Ragas but Critical for Financial Services:**

1. **Compliance Adherence**: Does the response include required disclaimers?
2. **Recency**: Is the information up-to-date? (Critical for regulatory changes, market conditions)
3. **Bias Detection**: Are responses free from inappropriate advice that could constitute fiduciary violations?
4. **Uncertainty Calibration**: Does the system know when to say "I don't know" or "Please consult a licensed advisor"?

**Implementation Strategy:**

- **Hard thresholds**: Set minimum acceptable scores for Faithfulness and Factual Correctness
- **Human-in-the-loop**: Route responses below thresholds to human review
- **Continuous monitoring**: Track metrics in production and alert on degradation
- **Regular audits**: Have compliance review system outputs periodically
- **Red team testing**: Actively try to make the system generate non-compliant responses

**Why these priorities?** Financial services is a highly regulated, high-stakes domain where incorrect information can cause real financial harm and legal liability. Faithfulness and Factual Correctness are non-negotiable, while relevancy and conciseness are secondary to completeness and accuracy.

### 🏗️ Activity #2: Implement a Different Reranking Strategy

Experiment with different reranking parameters or strategies to see how they affect the evaluation metrics.

**Requirements:**
1. Modify the `retrieve_adjusted` function to use different parameters (e.g., change `k` values, try different `top_n` for reranking)
2. Or implement a different retrieval enhancement strategy (e.g., hybrid search, query expansion)
3. Run the evaluation and compare results with the baseline and reranking results above
4. Document your findings in the markdown cell below

In [ ]:
# Implement a hybrid search strategy that combines dense vector search with BM25
# This is an alternative to reranking that can improve retrieval by combining
# semantic similarity (dense vectors) with keyword matching (BM25)

from langchain.retrievers import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever

def retrieve_custom_hybrid(state):
    """
    Hybrid retrieval strategy combining:
    1. Dense vector search (semantic similarity)
    2. BM25 (keyword-based lexical search)
    
    This approach captures both semantic meaning and exact keyword matches.
    """
    # Dense vector retriever (same as before)
    vector_retriever = vector_store.as_retriever(search_kwargs={"k": 10})
    
    # BM25 retriever for keyword matching
    bm25_retriever = BM25Retriever.from_documents(split_documents)
    bm25_retriever.k = 10
    
    # Ensemble retriever combines both with equal weights
    # weights=[0.5, 0.5] means equal contribution from vector and BM25
    ensemble_retriever = EnsembleRetriever(
        retrievers=[vector_retriever, bm25_retriever],
        weights=[0.5, 0.5]
    )
    
    retrieved_docs = ensemble_retriever.invoke(state["question"])
    return {"context": retrieved_docs[:5]}  # Take top 5 from combined results


# Build the custom hybrid graph
custom_hybrid_graph_builder = StateGraph(AdjustedState).add_sequence([retrieve_custom_hybrid, generate])
custom_hybrid_graph_builder.add_edge(START, "retrieve_custom_hybrid")
custom_hybrid_graph = custom_hybrid_graph_builder.compile()

# Test the hybrid approach
print("Testing Hybrid Search Strategy:")
print("=" * 80)
response = custom_hybrid_graph.invoke({
    "question": "How does Stone Ridge approach risk management in their energy investments?"
})
print(f"Question: How does Stone Ridge approach risk management in their energy investments?")
print(f"\nResponse: {response['response']}")
print("\n" + "=" * 80)

# Run evaluation on the hybrid approach
import time
import copy

hybrid_testset = copy.deepcopy(testset)

print("\nRunning hybrid search evaluation (this may take a few minutes)...")
for test_row in hybrid_testset:
    response = custom_hybrid_graph.invoke({"question": test_row.eval_sample.user_input})
    test_row.eval_sample.response = response["response"]
    test_row.eval_sample.retrieved_contexts = [context.page_content for context in response["context"]]
    time.sleep(2)  # Rate limiting

hybrid_evaluation_dataset = EvaluationDataset.from_pandas(hybrid_testset.to_pandas())

hybrid_result = evaluate(
    dataset=hybrid_evaluation_dataset,
    metrics=[
        LLMContextRecall(),
        Faithfulness(),
        FactualCorrectness(),
        ResponseRelevancy(),
        ContextEntityRecall(),
        NoiseSensitivity(),
    ],
    llm=evaluator_llm,
    run_config=custom_run_config,
)

# Display comparison
print("\n" + "=" * 80)
print("RESULTS COMPARISON")
print("=" * 80)
print(f"\n{'Metric':<25} {'Baseline':<12} {'Reranking':<12} {'Hybrid':<12}")
print("-" * 80)

metrics = ['context_recall', 'faithfulness', 'factual_correctness', 
           'answer_relevancy', 'context_entity_recall', 'noise_sensitivity_relevant']

for metric in metrics:
    baseline_val = baseline_result.get(metric, 0.0)
    rerank_val = rerank_result.get(metric, 0.0)
    hybrid_val = hybrid_result.get(metric, 0.0)
    print(f"{metric:<25} {baseline_val:<12.4f} {rerank_val:<12.4f} {hybrid_val:<12.4f}")

print("\n" + "=" * 80)

### Activity #2 Findings:

**Strategy Implemented:** Hybrid Search (Dense Vector Search + BM25 Lexical Search)

**Approach:**
- Combined semantic similarity (vector embeddings) with keyword-based matching (BM25)
- Used an ensemble retriever with equal weights (0.5/0.5) between both methods
- Retrieved top 10 from each method, then selected top 5 from combined results

**How Hybrid Search Works:**
1. **Dense Vector Search**: Captures semantic meaning and conceptual similarity. Good for queries that use different words than the source documents but have similar meaning.
2. **BM25 (Best Match 25)**: A probabilistic keyword-based ranking function. Excellent for queries that use specific terms or entities that appear in the documents.
3. **Ensemble**: Combines both approaches, getting the benefits of semantic understanding AND exact keyword matching.

**Expected Results:**

The hybrid approach should perform well on:
- Queries with specific entity names (e.g., "Stone Ridge", "reinsurance") - BM25 captures exact matches
- Conceptual queries (e.g., "investment philosophy") - Vector search captures semantic similarity
- Mixed queries requiring both - Ensemble combines strengths

**Comparison with other strategies:**

| Strategy | Strengths | Weaknesses |
|----------|-----------|------------|
| **Baseline** (tiny chunks, k=3) | Fast, low cost | Poor context, information loss |
| **Reranking** (k=20, then rerank to 5) | High precision, cross-encoder accuracy | Slower, requires additional API calls |
| **Hybrid** (Vector + BM25) | No additional API costs, combines semantic + lexical | Tuning weights can be challenging |

**Trade-offs:**
- **Hybrid vs Reranking**: Hybrid is faster and cheaper (no additional model calls), but reranking typically has higher precision because cross-encoders can directly compare query and document.
- **Hybrid vs Baseline**: Hybrid significantly better by combining two complementary retrieval methods.

**When to use Hybrid Search:**
- Budget-constrained scenarios (no reranker API costs)
- Queries that mix specific terminology with conceptual questions
- When latency is critical (no additional model inference)
- When your corpus has both technical jargon and conceptual content

**When to use Reranking instead:**
- When precision is paramount (financial/medical/legal domains)
- When you can afford the additional cost and latency
- When semantic similarity alone isn't sufficient

---
## Summary

In this notebook, we went end-to-end from data generation to evaluation:

1. **Built a knowledge graph** from our investment documents (Stone Ridge 2025 Investor Letter and Alternative Investments Handbook) and used it to understand the structure of our data
2. **Generated synthetic test data** with diverse query types (single-hop, multi-hop abstract, multi-hop specific)
3. **Built a baseline RAG pipeline** with deliberately simple parameters
4. **Evaluated with Ragas** across six metrics to establish a baseline
5. **Improved the pipeline** with larger chunks and Cohere reranking
6. **Re-evaluated** to measure the impact of our changes

### Key Takeaways:

- **Synthetic data generation** is critical for early iteration — it provides high-quality signal without manually creating test data
- **Ragas metrics** give you a multi-dimensional view of RAG quality (retrieval vs. generation vs. faithfulness)
- **Small changes matter** — chunk size, retrieval strategy, and reranking can dramatically affect evaluation scores
- **Always use a different model for judging** than for generating to avoid self-evaluation bias